In [ ]:
import numpy as np
import sklearn
%matplotlib inline
import matplotlib as mpl 
import matplotlib.pyplot as plt

The following commands are the same as for the Boosted Decision Tree file. They are used to extract the information from the dataset and to divide it into the necessary groups for training, validation and testing. 

In [ ]:
import pandas as pd

df = pd.read_csv('HIGGS.csv.gz', header=None, nrows=500_000)

In [ ]:
y = df.iloc[:, 0].values 
X = df.iloc[:, 1:].values

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

The first attempt to test the Deep Neural Network algorithms consists of a simple model with two layers, five epochs, and a 40% validation split. Below, "Sequential" defines the neural network as a sequence of steps; the "shape" parameter defines the structure of the array given in input, so, in this case, 28 features; while every layer is characterised by a number of neurons and by an activation function. The final, output, layer contains a single neuron, whose purpose is to classify each incoming event as either signal or background by producing an output value between 0 and 1. 

In [ ]:
from tensorflow.keras import models
from tensorflow.keras import Input, layers
higgs_DNN_0 = models.Sequential([
  Input(shape=(28,)), 
  layers.Dense(128, activation='relu'), 
  layers.Dense(1, activation='sigmoid') 
])

In [ ]:
higgs_DNN_0.summary()

What follows is the compilation step. The optimizer is the algorithm that decides how to update the weights after each batch of training, "adam" stands for "ADAptive Moment estimation" and is largely used mainly due to two main features: it tracks how much each weight has been changing and adjusts its learning rate accordingly, also remembering the direction of previous updates and using it to smooth the path toward the minimum. The other two terms define the mathematical error that the model tries to minimise while learning, in this case it is "binary_crossentropy" because the algorithm needs to distinguish between two categories: signal and background, while "metrics" allow for the loss and accuracy parameters to be visible in the output when the fit is executed. 

In [ ]:
higgs_DNN_0.compile(optimizer='adam', 
                   loss='binary_crossentropy', 
                   metrics=['accuracy']) 

For what concerns the fit process, the required arguments are the number of epochs, representing the number of times the entire training dataset is passed through the neural network; the batch size, indicating how many events the network sees before updating its weights once; and the validation split, which determines the subset of the training dataset that will be kept for internal testing. 

In [ ]:
%%time

history_0 = higgs_DNN_0.fit(X_train, y_train, epochs=5, batch_size=128, validation_split=0.4)

In [ ]:
test_loss, test_acc = higgs_DNN_0.evaluate(X_test, y_test)
print('test_acc:', test_acc)

Given the low accuracy on the test set a check that compared both the actual label of the event and its predicted one was performed. As it can be seen below, the model isn't capable of correctly identifying many of the events. 

In [ ]:
prob_labels = higgs_DNN_0.predict(X_test, verbose=0)
pred_labels = (prob_labels > 0.5).astype(int)

print("Actual label:     ", y_test[:20])
print("Predicted label:", pred_labels[:20])

The following plot is generated in order to better visualise the performance of the algorithm. It shows the values of the accuracy and loss on both the training and validation datasets in function of the number of epochs. 

In [ ]:
import pandas as pd

pd.DataFrame(history_0.history).plot()

Given the low performances of the model, a new one was developed with an increased number of layers, keeping the other parameters unchanged. 

In [ ]:
from tensorflow.keras import models
from tensorflow.keras import Input, layers
higgs_DNN_1 = models.Sequential([ 
  Input(shape=(28,)),  
  layers.Dense(128, activation='relu'), 
  layers.Dense(128, activation='relu'),
  layers.Dense(128, activation='relu'),
  layers.Dense(128, activation='relu'),
  layers.Dense(128, activation='relu'),
  layers.Dense(1, activation='sigmoid')
])

In [ ]:
higgs_DNN_1.compile(optimizer='adam', 
                   loss='binary_crossentropy', 
                   metrics=['accuracy']) 

In [ ]:
%%time

history_1 = higgs_DNN_1.fit(X_train, y_train, epochs=5, batch_size=128, validation_split=0.4)

In [ ]:
test_loss, test_acc = higgs_DNN_1.evaluate(X_test, y_test)
print('test_acc:', test_acc)

In [ ]:
pd.DataFrame(history_1.history).plot()

As visible from the values of the above results, increasing the number of layers didn't lead to a significant improvement in the model performance. 

For this reason, a new algorithm was built. This one kept the number of layers defined in the previous model, but decreased the validation split to the 30%, hoping in this way to allow for a better training of the model on the dataset. 

In [ ]:
from tensorflow.keras import models
from tensorflow.keras import Input, layers
higgs_DNN_2 = models.Sequential([
  Input(shape=(28,)),  
  layers.Dense(128, activation='relu'), 
  layers.Dense(128, activation='relu'),
  layers.Dense(128, activation='relu'),
  layers.Dense(128, activation='relu'),
  layers.Dense(128, activation='relu'),
  layers.Dense(1, activation='sigmoid')
])

In [ ]:
higgs_DNN_2.compile(optimizer='adam', 
                   loss='binary_crossentropy', 
                   metrics=['accuracy']) 

In [ ]:
%%time

history_2 = higgs_DNN_2.fit(X_train, y_train, epochs=5, batch_size=128, validation_split=0.3)

In [ ]:
test_loss, test_acc = higgs_DNN_2.evaluate(X_test, y_test)
print('test_acc:', test_acc)

In [ ]:
pd.DataFrame(history_2.history).plot()

This third model, characterised by a lower validation split, showed another small improvement in the accuracy with respect to the second one. Moreover, from the above plot it can be seen that this algorithm is not subject to overfitting, as it would lead to loss and validation loss curves, as well as accuracy and validation accuracy graphs, furthering apart one from the other. 

The next model has been built keeping the progressive improvements that had been obtained in the previous algorithms (increased number of layers and smaller validation split) and instead doubling the number of epochs from 5 to 10, hoping to reach a better training.  

In [ ]:
from tensorflow.keras import models
from tensorflow.keras import Input, layers
higgs_DNN_3 = models.Sequential([
  Input(shape=(28,)),  
  layers.Dense(128, activation='relu'), 
  layers.Dense(128, activation='relu'),
  layers.Dense(128, activation='relu'),
  layers.Dense(128, activation='relu'),
  layers.Dense(128, activation='relu'),
  layers.Dense(1, activation='sigmoid')
])

In [ ]:
higgs_DNN_3.compile(optimizer='adam', 
                   loss='binary_crossentropy', 
                   metrics=['accuracy']) 

In [ ]:
%%time

history_3 = higgs_DNN_3.fit(X_train, y_train, epochs=10, batch_size=128, validation_split=0.3)

In [ ]:
test_loss, test_acc = higgs_DNN_3.evaluate(X_test, y_test)
print('test_acc:', test_acc)

In [ ]:
pd.DataFrame(history_3.history).plot()

The above results show that increasing the number of epochs did indeed lead to an increase in the accuracy, although the plot could indicate some signs of overfitting of the model. 

To test the hypothesis that the number of epochs could be the cause of the overfitting and to also search for a model with a better accuracy, the next algorithm was developed doubling again the number of epochs (from 10 to 20). 

In [ ]:
from tensorflow.keras import models
from tensorflow.keras import Input, layers
higgs_DNN_4 = models.Sequential([
  Input(shape=(28,)),  
  layers.Dense(128, activation='relu'), 
  layers.Dense(128, activation='relu'),
  layers.Dense(128, activation='relu'),
  layers.Dense(128, activation='relu'),
  layers.Dense(128, activation='relu'),
  layers.Dense(1, activation='sigmoid')
])

In [ ]:
higgs_DNN_4.compile(optimizer='adam', 
                   loss='binary_crossentropy', 
                   metrics=['accuracy']) 

In [ ]:
%%time

history_4 = higgs_DNN_4.fit(X_train, y_train, epochs=20, batch_size=128, validation_split=0.3)

In [ ]:
test_loss, test_acc = higgs_DNN_4.evaluate(X_test, y_test)
print('test_acc:', test_acc)

In [ ]:
pd.DataFrame(history_4.history).plot()

The above plot seems to confirm the hypothesis: the number of epochs plays a role in the magnitude of the overfitting of the model. 

Another algorithm has been designed with the goal of finding a more accurate model. In this case the number of layers was increased again (passing from six to nine) and, instead, the epochs number has been set again to ten to avoid the overfitting effects seen previously. 

In [ ]:
from tensorflow.keras import models
from tensorflow.keras import Input, layers
higgs_DNN_5 = models.Sequential([
  Input(shape=(28,)),  
  layers.Dense(128, activation='relu'), 
  layers.Dense(128, activation='relu'),
  layers.Dense(128, activation='relu'),
  layers.Dense(128, activation='relu'),
  layers.Dense(128, activation='relu'),
  layers.Dense(128, activation='relu'), 
  layers.Dense(128, activation='relu'),
  layers.Dense(128, activation='relu'),
  layers.Dense(1, activation='sigmoid')
])

In [ ]:
higgs_DNN_5.compile(optimizer='adam', 
                   loss='binary_crossentropy', 
                   metrics=['accuracy'])

In [ ]:
%%time

history_5 = higgs_DNN_5.fit(X_train, y_train, epochs=10, batch_size=128, validation_split=0.3)

In [ ]:
test_loss, test_acc = higgs_DNN_5.evaluate(X_test, y_test)
print('test_acc:', test_acc)

In [ ]:
pd.DataFrame(history_5.history).plot()

This model shows a very scarce improvement in the test dataset accuracy with respect to the fourth algorithm, in this way not justifying the cost of increasing the number of layers. Moreover, signs of overfitting can be seen in the plot. 

The last two models have been developed using the same parameters of the fourth algorithm (higgs_DNN_3) and varying two important features that have been so far left untouched: the batch size in higgs_DNN_6 and the number of neurons in higgs_DNN_7. 

In [ ]:
from tensorflow.keras import models
from tensorflow.keras import Input, layers
higgs_DNN_6 = models.Sequential([
  Input(shape=(28,)),  
  layers.Dense(128, activation='relu'), 
  layers.Dense(128, activation='relu'),
  layers.Dense(128, activation='relu'),
  layers.Dense(128, activation='relu'),
  layers.Dense(128, activation='relu'),
  layers.Dense(1, activation='sigmoid')
])

In [ ]:
higgs_DNN_6.compile(optimizer='adam', 
                   loss='binary_crossentropy', 
                   metrics=['accuracy']) 

In [ ]:
%%time

history_6 = higgs_DNN_6.fit(X_train, y_train, epochs=10, batch_size=1024, validation_split=0.3)

In [ ]:
test_loss, test_acc = higgs_DNN_6.evaluate(X_test, y_test)
print('test_acc:', test_acc)

In [ ]:
pd.DataFrame(history_6.history).plot()

Increasing the batch size has led to a decrease in the accuracy and also to the appearance of overfitting signs, so actually worsening the original higgs_DNN_3 model. 

In [ ]:
from tensorflow.keras import models
from tensorflow.keras import Input, layers
higgs_DNN_7 = models.Sequential([
  Input(shape=(28,)),  
  layers.Dense(300, activation='relu'), 
  layers.Dense(300, activation='relu'),
  layers.Dense(1, activation='sigmoid') 
])

In [ ]:
higgs_DNN_7.compile(optimizer='adam', 
                   loss='binary_crossentropy', 
                   metrics=['accuracy']) 

In [ ]:
%%time

history_7 = higgs_DNN_7.fit(X_train, y_train, epochs=10, batch_size=128, validation_split=0.3)

In [ ]:
test_loss, test_acc = higgs_DNN_7.evaluate(X_test, y_test)
print('test_acc:', test_acc)

In [ ]:
pd.DataFrame(history_7.history).plot()

The neurons number seems to play an important role in the accuracy value: increasing it from 128 to 1024 actually increased the training, validation and test accuracies. However, in the above plot some indications of potential overfitting can be seen. 

As it has been done for the Boosted Decision Tree models, also in this case ROC curves and AUC values have been used to identify the best-performing model among the eight considered so far. The results of this analysis are presented and discussed below. 

In [ ]:
from sklearn.metrics import roc_curve, auc

models = [
    higgs_DNN_0,
    higgs_DNN_1,
    higgs_DNN_2,
    higgs_DNN_3,
    higgs_DNN_4,
    higgs_DNN_5,
    higgs_DNN_6,
    higgs_DNN_7
]

plt.figure(figsize=(8, 6))

for i, model in enumerate(models):
    y_scores_i = model.predict(X_test, verbose=0)
    fpr_i, tpr_i, thresholds_i = roc_curve(y_test, y_scores_i)
    auc_value_i = auc(fpr_i, tpr_i)
    plt.plot(
        fpr_i, 
        tpr_i, 
        label=f'DNN_{i} (AUC = {auc_value_i:.3f})'
    )
    
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate (background efficiency)')
plt.ylabel('True Positive Rate (signal efficiency)')
plt.title('ROC Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


Looking at the AUC values, it is strikingly evident the abrupt increase in efficiency passing from the first basic model (higgs_DNN_0) to the second one (higgs_DNN_1), where the only difference was related to the increase in the number of layers. Considering just the above plot, the best-performing model could be the fifth one (higgs_DNN_4), characterised by having the largest number of epochs with respect to the other algorithms (20). However, that same model showed also dramatic signs of overfitting, therefore showing that the choice of the model to use should take into consideration different parameters and not just one. For this reason, the algorithm which seems to have not only a number of layers and neurons which justify the obtained values of accuracy and AUC, but also no signs of overfitting, is the third one (higgs_DNN_2), characterised by six layers, each with 128 neurons, and "epochs" = 5, "batch_size" = 128, "validation_split" = 0.3. 

For better visualisation, the ROC curves for the five models with the highest AUC values (higgs_DNN_2, higgs_DNN_3, higgs_DNN_4, higgs_DNN_5, higgs_DNN_7) are visible below.

In [ ]:
from sklearn.metrics import roc_curve, auc

models = [
    (2, higgs_DNN_2),
    (3, higgs_DNN_3),
    (4, higgs_DNN_4),
    (5, higgs_DNN_5),
    (7, higgs_DNN_7)
]

plt.figure(figsize=(8, 6))

for dnn_number, model in models:
    y_scores = model.predict(X_test, verbose=0)
    
    fpr, tpr, thresholds = roc_curve(y_test, y_scores)
    auc_value = auc(fpr, tpr)
    
    plt.plot(
        fpr,
        tpr,
        label=f'DNN_{dnn_number} (AUC = {auc_value:.3f})'
    )

plt.plot([0, 1], [0, 1], 'k--', label='Random')

plt.xlabel('False Positive Rate (background efficiency)')
plt.ylabel('True Positive Rate (signal efficiency)')
plt.title('ROC Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

Recalling the analysis of the feature importance performed during the training of the models using Boosted Decision Trees, which showed that high-level features have a significantly greater impact on the training than low-level features, the same approach is applied here to Deep Neural Networks. In this case as well, the analysis is performed using the algorithm that achieved the best performance in the previous study, namely higgs_DNN_2. 

In [ ]:
y_l = df.iloc[:, 0].values         
X_l = df.iloc[:, 1:22].values 

In [ ]:
from sklearn.model_selection import train_test_split

X_l_train_full, X_l_test, y_l_train_full, y_l_test = train_test_split(X_l, y_l, test_size=0.2, random_state=42) 

X_l_train, X_l_val, y_l_train, y_l_val = train_test_split(X_l_train_full, y_l_train_full, test_size=0.10, random_state=42) 

In [ ]:
from tensorflow.keras import models
from tensorflow.keras import Input, layers
higgs_DNN_2_l = models.Sequential([
  Input(shape=(21,)),  
  layers.Dense(128, activation='relu'), 
  layers.Dense(128, activation='relu'),
  layers.Dense(128, activation='relu'),
  layers.Dense(128, activation='relu'),
  layers.Dense(128, activation='relu'),
  layers.Dense(1, activation='sigmoid')
])

In [ ]:
higgs_DNN_2_l.compile(optimizer='adam', 
                   loss='binary_crossentropy', 
                   metrics=['accuracy']) 

In [ ]:
%%time

history_2_l = higgs_DNN_2_l.fit(X_l_train, y_l_train, epochs=5, batch_size=128, validation_split=0.3)

In [ ]:
test_loss, test_acc = higgs_DNN_2_l.evaluate(X_l_test, y_l_test)
print('test_acc:', test_acc)

In [ ]:
y_h = df.iloc[:, 0].values         
X_h = df.iloc[:, 22:].values 

In [ ]:
from sklearn.model_selection import train_test_split

X_h_train_full, X_h_test, y_h_train_full, y_h_test = train_test_split(X_h, y_h, test_size=0.2, random_state=42) 

X_h_train, X_h_val, y_h_train, y_h_val = train_test_split(X_h_train_full, y_h_train_full, test_size=0.10, random_state=42)

In [ ]:
from tensorflow.keras import models
from tensorflow.keras import Input, layers
higgs_DNN_2_h = models.Sequential([
  Input(shape=(7,)),  
  layers.Dense(128, activation='relu'), 
  layers.Dense(128, activation='relu'),
  layers.Dense(128, activation='relu'),
  layers.Dense(128, activation='relu'),
  layers.Dense(128, activation='relu'),
  layers.Dense(1, activation='sigmoid')
])

In [ ]:
higgs_DNN_2_h.compile(optimizer='adam', 
                   loss='binary_crossentropy', 
                   metrics=['accuracy']) 

In [ ]:
%%time

history_2_h = higgs_DNN_2_h.fit(X_h_train, y_h_train, epochs=5, batch_size=128, validation_split=0.3)

In [ ]:
test_loss, test_acc = higgs_DNN_2_h.evaluate(X_h_test, y_h_test)
print('test_acc:', test_acc)

In [ ]:
from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(8, 6))

y_scores = higgs_DNN_2.predict(X_test, verbose=0)
fpr, tpr, thresholds = roc_curve(y_test, y_scores)
auc_value = auc(fpr, tpr)
plt.plot(
    fpr,
    tpr,
    label=f'DNN_2 (AUC = {auc_value:.3f})'
)

y_scores = higgs_DNN_2_l.predict(X_l_test, verbose=0) 
fpr, tpr, thresholds = roc_curve(y_l_test, y_scores)
auc_value = auc(fpr, tpr)
plt.plot(
    fpr,
    tpr,
    label=f'DNN_2_l (AUC = {auc_value:.3f})'
)

y_scores = higgs_DNN_2_h.predict(X_h_test, verbose=0) 
fpr, tpr, thresholds = roc_curve(y_h_test, y_scores)
auc_value = auc(fpr, tpr)
plt.plot(
    fpr,
    tpr,
    label=f'DNN_2_h (AUC = {auc_value:.3f})'
)

plt.plot([0, 1], [0, 1], 'k--', label='Random')

plt.xlabel('False Positive Rate (background efficiency)')
plt.ylabel('True Positive Rate (signal efficiency)')
plt.title('ROC Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

The ROC curves and corresponding AUC values show a behaviour consistent with that observed for Boosted Decision Trees: high-level features play a significant role in the training and performance of the models. In the case of Deep Neural Networks, their contribution appears to be even more pronounced, as indicated by the small difference in AUC between the model trained exclusively on high-level features and the model trained using the complete feature set.